### Agentic RAG

In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper

C:\Users\Rupesh\AppData\Local\Temp\ipykernel_20916\2309859512.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
@tool("GoogleSearch")
def search(query_string: str):
    """
    Useful to search for any kinds of information and
    when you need to search the internet for any kinds of information, use this tool.
    Prefer this tool when you search for long queries.
    Should not be used for Article search or Topic Search.
    """
    
    search = GoogleSerperAPIWrapper()
    
    return search.run(query_string)

In [3]:

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wiki = WikipediaQueryRun(
    name="WikiepdiaSearch",
    description="Use this tool when you want to analyze for information on Wikipedia by Terms, Keywords or any Topics.",
    api_wrapper=api_wrapper)

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

c:\Users\Rupesh\Desktop\learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
# Load the webpage
loader = WebBaseLoader("https://docs.smith.langchain.com")
docs = loader.load()

# Split the documents into chunks
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

# Create Hugging Face embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create the FAISS vector store
vectordatabase = FAISS.from_documents(
    documents,
    embeddings
)

# Create the retriever
retriever = vectordatabase.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3156.19it/s]


In [7]:
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="langsmith_search",
    description=(
        "Search for information about LangSmith. "
        "Use this tool whenever a question is related to LangSmith."
    )
)

print(retriever_tool.name)

langsmith_search


In [8]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

In [9]:
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

In [10]:
tools = [arxiv, search, wiki, retriever_tool]
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=1000)),
 StructuredTool(name='GoogleSearch', description='Useful to search for any kinds of information and\nwhen you need to search the internet for any kinds of information, use this tool.\nPrefer this tool when you search for long queries.\nShould not be used for Article search or Topic Search.', args_schema=<class 'langchain_core.utils.pydantic.GoogleSearch'>, func=<function search at 0x000001E21D8CF1A0>),
 WikipediaQueryRun(name='WikiepdiaSearch', description='Use this tool when you want to analyze for information on Wikipedia by Terms, Keywords or any Topics.', api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Rupes

In [11]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [12]:
load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or "llama-3.1-8b-instant"
    max_tokens=2000,
    temperature=0.1,
    api_key=os.environ["GROQ_API_KEY"]
)

In [17]:
from langchain import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

c:\Users\Rupesh\Desktop\learning\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)
c:\Users\Rupesh\Desktop\learning\.venv\Lib\site-packages\langchain\hub.py:87: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  obj = loads(json.dumps(res_dict["manifest"]))


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs=FieldInfo(default=PydanticUndefined, default_factory=<class 'dict'>, extra={})),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs=FieldInfo(default=PydanticUndefined, default_factory=<class 'dict'>, extra={})),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [18]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage

agent_executor = create_react_agent(
    llm,
    tools = tools
)

C:\Users\Rupesh\AppData\Local\Temp\ipykernel_20916\3333576309.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [20]:
for stream in agent_executor.stream({
    "messages": [
        HumanMessage(content="tell me about langsmith")
    ]
}):
    print(stream)
    print("***********")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9shvbx5fa', 'function': {'arguments': '{"query":"LangSmith information"}', 'name': 'langsmith_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 635, 'total_tokens': 653, 'completion_time': 0.057467767, 'completion_tokens_details': None, 'prompt_time': 0.049994196, 'prompt_tokens_details': None, 'queue_time': 0.052631104, 'total_time': 0.107461963}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8345-4ec9-7b20-9eee-ba281fe300da-0', tool_calls=[{'name': 'langsmith_search', 'args': {'query': 'LangSmith information'}, 'id': '9shvbx5fa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 635, 'output_tokens': 18, 'total_tokens': 653})]}}
***********
{'tools': {'messag

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kjcjp5fter7rkw4xe65bt76a` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 93455, Requested 9187. Please try again in 38m2.688s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [17]:
for stream in agent_executor.stream({
    "messages": [
        HumanMessage(content="whats the paper 2412.16446 talk about it?")
    ]
}):
    print(stream)
    print("***********")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kq8s2SI08WW7gR1fOelFrLr1', 'function': {'arguments': '{"query":"2412.16446"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 264, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': None, 'id': 'chatcmpl-BtXKT5Fsor9pi6HDLOMtBCiFtXpLa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--7db40a4b-d2ef-4cb6-84bd-e3337d252e9c-0', tool_calls=[{'name': 'arxiv', 'args': {'query': '2412.16446'}, 'id': 'call_Kq8s2SI08WW7gR1fOelFrLr1', 'type': 'tool_call'}], usage_metadata={'input_tokens': 264, 'output_tokens': 18, 'total_tokens': 282, 'input_token_detail

In [20]:
for stream in agent_executor.stream({
    "messages": [
        HumanMessage(content="Indian Constitution")
    ]
}):
    print(stream)
    print("***********")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_T4DEdEIiuF5zUemY5rqj40eE', 'function': {'arguments': '{"query":"Indian Constitution"}', 'name': 'WikiepdiaSearch'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 252, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': None, 'id': 'chatcmpl-BtXMD6QKEBOIN5zZHKTSK4fFwd9Gk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--57f96da2-761c-4e39-a223-03589bd86fc6-0', tool_calls=[{'name': 'WikiepdiaSearch', 'args': {'query': 'Indian Constitution'}, 'id': 'call_T4DEdEIiuF5zUemY5rqj40eE', 'type': 'tool_call'}], usage_metadata={'input_tokens': 252, 'output_tokens': 19, 't

In [22]:
for stream in agent_executor.stream({
    "messages": [
        HumanMessage(content="Who won the world cup in the Year 2025 in WTC?")
    ]
}):
    print(stream)
    print("***********")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_6BIoSj9ra1f7QLTOnCJae64X', 'function': {'arguments': '{"query_string":"2025 World Test Championship winner"}', 'name': 'GoogleSearch'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 265, 'total_tokens': 285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': None, 'id': 'chatcmpl-BtXNHaXXSoE4Q0Bw6TkdnmGOPUQmV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--a7ff6041-d867-45a0-abc0-5688aa24fada-0', tool_calls=[{'name': 'GoogleSearch', 'args': {'query_string': '2025 World Test Championship winner'}, 'id': 'call_6BIoSj9ra1f7QLTOnCJae64X', 'type': 'tool_call'}], usage_metadata={'inp